# 12 Constraints, Generated, Identity and Default Columns

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Enforce data rules inside the table with <code>NOT NULL</code> and <code>CHECK</code> constraints, compute columns automatically with <b>generated columns</b>, create surrogate keys with <b>identity columns</b>, set <b>default values</b>, and document relationships with informational primary and foreign keys in Unity Catalog.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Rules enforced by the table protect it from <b>every</b> writer: your pipeline, an analyst's notebook, a colleague's quick fix. Generated and identity columns remove whole classes of bugs (a forgotten <code>year</code> column, duplicate surrogate keys). These features appear in exam questions on data quality and table design.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Table-level rules and helpers</b><br>
| Feature | Enforced? | Example |<br>|---|---|---|<br>| <code>NOT NULL</code> | ✅ every write | <code>order_id BIGINT NOT NULL</code> |<br>| <code>CHECK</code> constraint | ✅ every write. Existing rows must also satisfy it when added | <code>CONSTRAINT positive_amount CHECK (amount &gt;= 0)</code> |<br>| Generated column | ✅ computed (or validated) on every write | <code>order_year INT GENERATED ALWAYS AS (year(order_date))</code> |<br>| Identity column | ✅ unique, increasing values assigned by Delta | <code>id BIGINT GENERATED ALWAYS AS IDENTITY</code> |<br>| Default value | Applied when a column is omitted | <code>status STRING DEFAULT 'new'</code> |<br>| Primary / foreign key (Unity Catalog) | ❌ <b>informational only</b> | <code>CONSTRAINT pk PRIMARY KEY (id)</code> |<br><br>
A write that violates <code>NOT NULL</code> or a <code>CHECK</code> fails <b>as a whole</b>: no rows are written.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An e-commerce Silver table once received negative quantities from a buggy mobile app release, and Gold inventory numbers went wrong for a week. The team added <code>CHECK (quantity &gt; 0)</code> to the table. The next bad release failed its load immediately and loudly, and their quarantine logic (lesson 01_29) routed the bad rows aside before writing.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sets the schema for this lesson.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

def run(sql, label):
    try:
        spark.sql(sql)
        print(f"✅ {label}")
    except Exception as e:
        print(f"❌ {label}: {getattr(e, 'getCondition', lambda: None)() or type(e).__name__}")

## 1. NOT NULL and CHECK constraints

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an <code>order_lines</code> table with <code>NOT NULL</code> columns and two <code>CHECK</code> constraints, then tries valid and invalid inserts.<br><br>
<b>Why it matters</b><br>Constraints are stored as table properties (<code>delta.constraints.&lt;name&gt;</code>), so they are visible to everyone and enforced by every engine that writes Delta.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The valid insert succeeds. The negative quantity, the invalid status and the null key are each rejected with a constraint violation. The table still has only the valid rows.
</div>

In [0]:
lines = f"{DB}.order_lines"
spark.sql(f"""
    CREATE OR REPLACE TABLE {lines} (
        order_id BIGINT NOT NULL,
        line_no  INT    NOT NULL,
        quantity INT,
        unit_price DECIMAL(10,2),
        status   STRING
    )
""")
spark.sql(f"ALTER TABLE {lines} ADD CONSTRAINT positive_quantity CHECK (quantity > 0)")
spark.sql(f"ALTER TABLE {lines} ADD CONSTRAINT valid_status CHECK (status IN ('new', 'shipped', 'returned'))")

run(f"INSERT INTO {lines} VALUES (1, 1, 2, 19.99, 'new'), (1, 2, 1, 5.00, 'shipped')", "valid rows")
run(f"INSERT INTO {lines} VALUES (2, 1, -3, 9.99, 'new')", "negative quantity")
run(f"INSERT INTO {lines} VALUES (3, 1, 1, 9.99, 'lost')", "unknown status")
run(f"INSERT INTO {lines} VALUES (NULL, 1, 1, 9.99, 'new')", "null order_id")
spark.table(lines).show()

## 2. Constraints are all or nothing, and checked against existing data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Inserts a batch where one row out of three is invalid, then tries to add a constraint that existing data violates.<br><br>
<b>Why it matters</b><br>A single bad row fails the whole write, so pair constraints with upstream validation and quarantine, or a whole batch is blocked. Adding a constraint that current data breaks also fails, so clean the data first.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The 3-row batch is rejected and the table is unchanged (2 rows). <code>ADD CONSTRAINT price_under_10</code> fails, because a row has a price of 19.99.
</div>

In [0]:
run(f"INSERT INTO {lines} VALUES (4, 1, 1, 3.00, 'new'), (4, 2, 0, 3.00, 'new'), (4, 3, 2, 3.00, 'new')", "batch with one bad row")
print("rows in table:", spark.table(lines).count())
run(f"ALTER TABLE {lines} ADD CONSTRAINT price_under_10 CHECK (unit_price < 10)", "constraint that existing data violates")

## 3. Inspecting and dropping constraints

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the constraints from the table properties, then drops one.<br><br>
<b>Why it matters</b><br>Constraints are part of the table's documented contract. You can see them with <code>SHOW TBLPROPERTIES</code> or <code>DESCRIBE DETAIL</code>, and remove them with <code>DROP CONSTRAINT</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two <code>delta.constraints.*</code> properties, then one after the drop.
</div>

In [0]:
spark.sql(f"SHOW TBLPROPERTIES {lines}").filter("key LIKE 'delta.constraints%'").show(truncate=False)
spark.sql(f"ALTER TABLE {lines} DROP CONSTRAINT valid_status")
spark.sql(f"SHOW TBLPROPERTIES {lines}").filter("key LIKE 'delta.constraints%'").show(truncate=False)

## 4. Generated columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a table where <code>order_year</code> and <code>order_month</code> are computed from <code>order_ts</code> automatically, then inserts rows without them.<br><br>
<b>Why it matters</b><br>Generated columns are always consistent with their source column. If a writer supplies a value, Delta checks that it matches the expression. They're commonly used for partition-like date columns and for derived keys.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Generated, identity and default columns are created with Delta's SQL DDL on Databricks. The cells below use that syntax.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Rows with <code>order_year = 2024</code> and <code>order_month = 11</code> computed automatically. An insert that supplies a wrong <code>order_year</code> is rejected.
</div>

In [0]:
events = f"{DB}.events_generated"
spark.sql(f"""
    CREATE OR REPLACE TABLE {events} (
        event_id    BIGINT,
        order_ts    TIMESTAMP,
        order_year  INT GENERATED ALWAYS AS (year(order_ts)),
        order_month INT GENERATED ALWAYS AS (month(order_ts))
    )
""")
spark.sql(f"INSERT INTO {events} (event_id, order_ts) VALUES (1, TIMESTAMP'2024-11-05 10:00:00'), (2, TIMESTAMP'2024-11-20 18:30:00')")
display(spark.table(events))
run(f"INSERT INTO {events} VALUES (3, TIMESTAMP'2024-12-01 09:00:00', 2023, 12)", "wrong generated value")

## 5. Identity columns: surrogate keys

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a dimension with an identity column, inserts rows without ids, and shows the generated values.<br><br>
<b>Why it matters</b><br>Surrogate keys must be unique. Generating them with <code>row_number()</code> or <code>monotonically_increasing_id()</code> breaks across runs or isn't consecutive. Identity columns are assigned by Delta and are <b>unique and increasing</b>, but <b>not necessarily consecutive</b> (gaps are normal, because of parallel writers).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Customers with automatically assigned <code>customer_sk</code> values (for example 1, 2, 3, possibly with gaps). Inserting an explicit value into a <code>GENERATED ALWAYS</code> identity column fails.
</div>

In [0]:
dim = f"{DB}.dim_customer_identity"
spark.sql(f"""
    CREATE OR REPLACE TABLE {dim} (
        customer_sk BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
        customer_id STRING,
        name        STRING
    )
""")
spark.sql(f"INSERT INTO {dim} (customer_id, name) VALUES ('C1', 'Amal'), ('C2', 'John'), ('C3', 'Vivek')")
display(spark.table(dim).orderBy("customer_sk"))
run(f"INSERT INTO {dim} VALUES (100, 'C4', 'Sara')", "explicit value into GENERATED ALWAYS identity")

## 6. Default values

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Enables column defaults, creates columns with <code>DEFAULT</code>, and inserts rows without those columns.<br><br>
<b>Why it matters</b><br>Defaults make the table's behaviour explicit instead of relying on every writer to fill a value. They apply when the column is omitted from an <code>INSERT</code> (or <code>DEFAULT</code> is written explicitly).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
New rows get <code>status = 'new'</code> and a <code>created_at</code> timestamp automatically.
</div>

In [0]:
tickets = f"{DB}.tickets_defaults"
spark.sql(f"""
    CREATE OR REPLACE TABLE {tickets} (
        ticket_id  BIGINT,
        subject    STRING,
        status     STRING    DEFAULT 'new',
        created_at TIMESTAMP DEFAULT current_timestamp()
    )
    TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
""")
spark.sql(f"INSERT INTO {tickets} (ticket_id, subject) VALUES (1, 'Login issue'), (2, 'Refund')")
display(spark.table(tickets))

## 7. Informational primary and foreign keys (Unity Catalog)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Declares a primary key on the customer dimension and a foreign key from an orders fact table.<br><br>
<b>Why it matters</b><br>In Unity Catalog, primary and foreign keys are <b>not enforced</b>: duplicates aren't rejected. They document relationships for people and tools (Catalog Explorer, BI tools, Genie), and with the <code>RELY</code> option the optimizer can use them for some query rewrites. Enforce uniqueness yourself, with <code>MERGE</code> and data quality checks.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The constraints appear in <code>DESCRIBE TABLE EXTENDED</code>. A duplicate key insert is <b>accepted</b>, which shows the keys are informational.
</div>

In [0]:
pk_dim = f"{DB}.dim_store_keys"
fact = f"{DB}.fact_sales_keys"
spark.sql(f"CREATE OR REPLACE TABLE {pk_dim} (store_id INT NOT NULL, city STRING, CONSTRAINT pk_store PRIMARY KEY (store_id))")
spark.sql(f"""
    CREATE OR REPLACE TABLE {fact} (
        sale_id BIGINT NOT NULL, store_id INT, amount DECIMAL(10,2),
        CONSTRAINT pk_sale PRIMARY KEY (sale_id),
        CONSTRAINT fk_store FOREIGN KEY (store_id) REFERENCES {pk_dim}
    )
""")
spark.sql(f"INSERT INTO {pk_dim} VALUES (1, 'Chennai'), (1, 'Duplicate!')")
print("duplicate primary key accepted, rows:", spark.table(pk_dim).count())
display(spark.sql(f"DESCRIBE TABLE EXTENDED {fact}").filter("col_name LIKE '%constraint%' OR col_name LIKE 'fk_%' OR col_name LIKE 'pk_%'"))

## Under the hood

```
ALTER TABLE ... ADD CONSTRAINT c CHECK (expr)
  1. scan existing data: any row violating expr? -> fail
  2. commit: metaData property delta.constraints.c = "expr" (+ protocol upgrade if needed)

INSERT / UPDATE / MERGE on the table
  - Delta adds the checks to the write plan: NOT NULL, CHECK expressions, generated-column expressions
  - any violating row -> the whole transaction fails, nothing is committed
Identity: Delta tracks the high-water mark in the table metadata and hands out ranges to writers
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> checks run during the write, before the commit.

Constraints, generated and identity columns require newer **table features** (protocol versions). Older readers or writers that don't support them can't write to the table.

In [0]:
spark.sql(f"DESCRIBE DETAIL {lines}").select("minReaderVersion", "minWriterVersion", "properties").show(truncate=False)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {lines}")

In [0]:
for t in [events, dim, tickets, fact, pk_dim]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>DELTA_VIOLATE_CONSTRAINT_WITH_VALUES</code> fails the whole batch</b><br>
Validate and quarantine bad rows before writing (lesson 01_29). The constraint is the last line of defence, not the first.<br><br>
**⛔ Problem: <code>ADD CONSTRAINT</code> fails**<br>Existing rows violate it. Fix or delete them first.<br><br>
<b>⛔ Problem: identity values have gaps</b><br>That's expected: identity values are unique and increasing, not consecutive.<br><br>
<b>⛔ Problem: duplicate primary keys exist</b><br>Unity Catalog keys are informational. Enforce uniqueness with <code>MERGE</code> and checks.<br><br>
<b>⛔ Problem: generated column value rejected</b><br>A writer supplied a value that doesn't match the expression. Omit the column and let Delta compute it.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What constraints does Delta Lake enforce?</b><br>
<code>NOT NULL</code> and <code>CHECK</code> constraints are enforced on every write, and a violating write fails entirely. Primary and foreign keys in Unity Catalog are informational and not enforced.<br><br>

<b>🎤 2. What is a generated column?</b><br>
A column whose value is computed from other columns by an expression declared in the table, such as <code>GENERATED ALWAYS AS (year(order_ts))</code>. Delta computes it on write, or validates it if a writer supplies a value.<br><br>

<b>🎤 3. How do you create surrogate keys in Delta?</b><br>
With an identity column, <code>BIGINT GENERATED ALWAYS AS IDENTITY</code>. Delta assigns unique, increasing values. They can have gaps, so don't rely on them being consecutive.<br><br>

<b>🎤 4. What happens when you add a <code>CHECK</code> constraint to a table with existing invalid rows?</b><br>
The <code>ALTER TABLE</code> fails. Delta validates existing data when the constraint is added.<br><br>

<b>🎤 5. Why use table constraints if the pipeline already validates data?</b><br>
They protect the table from every writer, including ad hoc notebooks and other jobs, and they document the rules in the table itself. Pipeline checks handle quarantine and reporting, and constraints are the final guarantee.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A Delta table has <code>CONSTRAINT valid_qty CHECK (qty &gt; 0)</code>. A batch of 1,000 rows contains 1 row with <code>qty = 0</code>. What happens when the batch is appended?</b><br>
A. 999 rows are written and 1 is dropped<br>
B. The whole write fails and no rows are added<br>
C. The invalid row is written with <code>qty</code> set to null<br>
D. The constraint is automatically disabled<br>
<details><summary><b>Show answer</b></summary><b>B.</b> To keep the valid rows, filter or quarantine before writing. Lakeflow expectations offer drop and warn behaviours.</details><br><br>

<b>Q2. Which column definition gives automatically assigned, unique surrogate keys in a Delta table?</b><br>
A. <code>id BIGINT DEFAULT monotonically_increasing_id()</code><br>
B. <code>id BIGINT GENERATED ALWAYS AS IDENTITY</code><br>
C. <code>id BIGINT PRIMARY KEY</code><br>
D. <code>id BIGINT GENERATED ALWAYS AS (row_number())</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Primary keys in Unity Catalog are informational and don't generate values.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create <code>payments (payment_id BIGINT NOT NULL, amount DECIMAL(10,2), currency STRING)</code> with constraints: amount positive, currency one of INR, GBP, AED, USD</li><li>Show that a batch with one invalid currency is rejected completely</li><li>Write PySpark code that splits an incoming DataFrame into valid rows (appended) and invalid rows (quarantined in a separate table) <b>using the same rules</b>, so the append never fails</li><li>On Databricks: add a generated column <code>amount_usd_cents BIGINT GENERATED ALWAYS AS (CAST(amount * 100 AS BIGINT))</code> to a new version of the table</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
pay = f"{DB}.payments_rules"
quarantine = f"{DB}.payments_quarantine"

# 1. Constraints
spark.sql(f"CREATE OR REPLACE TABLE {pay} (payment_id BIGINT NOT NULL, amount DECIMAL(10,2), currency STRING)")
spark.sql(f"ALTER TABLE {pay} ADD CONSTRAINT positive_amount CHECK (amount > 0)")
spark.sql(f"ALTER TABLE {pay} ADD CONSTRAINT known_currency CHECK (currency IN ('INR', 'GBP', 'AED', 'USD'))")

# 2. One bad currency rejects the whole batch
run(f"INSERT INTO {pay} VALUES (1, 10.00, 'INR'), (2, 5.00, 'EUR')", "batch with EUR")

# 3. Same rules in code: append the valid rows, quarantine the rest
incoming = spark.createDataFrame([(3, 12.5, "GBP"), (4, -1.0, "USD"), (5, 7.0, "JPY")], "payment_id BIGINT, amount DOUBLE, currency STRING") \
    .withColumn("amount", F.col("amount").cast("decimal(10,2)"))
is_valid = (F.col("payment_id").isNotNull() & (F.col("amount") > 0) & F.col("currency").isin("INR", "GBP", "AED", "USD"))
incoming.filter(is_valid).write.mode("append").saveAsTable(pay)
incoming.filter(~is_valid).withColumn("quarantined_at", F.current_timestamp()).write.mode("overwrite").saveAsTable(quarantine)
spark.table(pay).show(); spark.table(quarantine).show()
for t in [pay, quarantine]:
    spark.sql(f"DROP TABLE {t}")

In [0]:
# 4. Generated column on a new version of the table (Databricks)
spark.sql(f"""
    CREATE OR REPLACE TABLE {pay} (
        payment_id BIGINT NOT NULL,
        amount DECIMAL(10,2),
        currency STRING,
        amount_usd_cents BIGINT GENERATED ALWAYS AS (CAST(amount * 100 AS BIGINT))
    )
""")
spark.sql(f"INSERT INTO {pay} (payment_id, amount, currency) VALUES (1, 12.34, 'USD')")
display(spark.table(pay))
spark.sql(f"DROP TABLE {pay}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>NOT NULL</code> and <code>CHECK</code> are enforced on every write. A violation fails the whole transaction</li><li>Adding a <code>CHECK</code> validates existing data first</li><li>Generated columns compute values from other columns (or validate supplied values)</li><li>Identity columns give unique, increasing surrogate keys (gaps are normal)</li><li><code>DEFAULT</code> values apply when a column is omitted (enable <code>allowColumnDefaults</code>)</li><li>Unity Catalog primary and foreign keys are informational, not enforced</li><li>Pair constraints with pipeline-side validation and quarantine</li></ul>
</div>

| Task | SQL |
|---|---|
| Not null | `col INT NOT NULL` |
| Check | `ALTER TABLE t ADD CONSTRAINT c CHECK (expr)` |
| Drop check | `ALTER TABLE t DROP CONSTRAINT c` |
| Generated | `y INT GENERATED ALWAYS AS (year(ts))` |
| Identity | `id BIGINT GENERATED ALWAYS AS IDENTITY` |
| Default | `status STRING DEFAULT 'new'` |
| Keys (UC) | `CONSTRAINT pk PRIMARY KEY (id)`, `FOREIGN KEY (x) REFERENCES t2` |

## Git commit

```
git add 03_delta_lake/12_constraints_generated_columns.ipynb
git commit -m "add 03_12 constraints generated columns notebook"
```